# 🧅 Onion Quality AI — YOLO11n Training (Google Colab T4 GPU)

**4 classes**: `onion` (healthy), `damaged`, `rotten`, `sprouted`

**Training time**: ~30–45 min on T4 GPU | **GO/NO-GO gate**: onion mAP50 ≥ 0.6

---

## Step 1 — Verify GPU & Install

In [ ]:
!nvidia-smi
!pip install -q ultralytics

## Step 2 — Upload & Unzip Dataset

Upload `onion_dataset.zip` (generated locally by `python ml/scripts/export_dataset.py`) using the Colab file browser sidebar, then run:

In [ ]:
!unzip -qo onion_dataset.zip -d /content/dataset

# Verify structure
import os
for split in ['train', 'val']:
    imgs = len(os.listdir(f'/content/dataset/images/{split}'))
    lbls = len(os.listdir(f'/content/dataset/labels/{split}'))
    print(f'{split}: {imgs} images, {lbls} labels')

## Step 3 — Write `data.yaml` (absolute paths for Colab)

In [ ]:
import yaml

data_cfg = {
    'path': '/content/dataset',
    'train': 'images/train',
    'val': 'images/val',
    'nc': 4,
    'names': ['onion', 'damaged', 'rotten', 'sprouted'],
}

with open('/content/dataset/data.yaml', 'w') as f:
    yaml.dump(data_cfg, f)

print('data.yaml written ✓')
print(yaml.dump(data_cfg))

## Step 4 — Label Distribution Check

In [ ]:
from collections import Counter
from pathlib import Path

CLASS_NAMES = {0: 'onion', 1: 'damaged', 2: 'rotten', 3: 'sprouted'}
counts = Counter()

for split in ['train', 'val']:
    for lbl in Path(f'/content/dataset/labels/{split}').glob('*.txt'):
        for line in lbl.read_text().strip().splitlines():
            cls_id = int(line.split()[0])
            counts[cls_id] += 1

total = sum(counts.values())
print(f'Total annotations: {total}')
print('-' * 40)
for cls_id in sorted(counts):
    name = CLASS_NAMES.get(cls_id, f'unknown-{cls_id}')
    pct = counts[cls_id] / total * 100
    bar = '█' * int(pct / 2)
    print(f'{cls_id} {name:>10}: {counts[cls_id]:>5} ({pct:5.1f}%) {bar}')

# Flag thin classes
for cls_id, count in counts.items():
    if count / total < 0.08:
        print(f'\n⚠️  WARNING: {CLASS_NAMES[cls_id]} has only {count} boxes ({count/total*100:.1f}%) — consider augmenting!')

## Step 5 — Train YOLO11n

~100 epochs on T4 GPU ≈ 30–45 minutes. `seed=42` for reproducibility.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')  # auto-downloads pretrained weights

results = model.train(
    data='/content/dataset/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    seed=42,
    name='onion_yolo_v1',
    # Augmentation tuned for onion photos
    hsv_h=0.015,   # hue variation (onions have narrow hue range)
    hsv_s=0.7,     # saturation variation
    hsv_v=0.4,     # brightness variation
    degrees=10.0,  # slight rotation
    translate=0.1,  # small translation
    scale=0.5,     # scale variation (close-up vs wide shots)
    fliplr=0.5,    # horizontal flip
)

## Step 6 — Evaluate & GO/NO-GO Gate

**Gate**: onion class mAP50 ≥ 0.6 (defect classes expected lower on this data size — fine for MVP)

In [ ]:
# Load best weights and evaluate
best_model = YOLO('/content/runs/detect/onion_yolo_v1/weights/best.pt')
metrics = best_model.val(data='/content/dataset/data.yaml', imgsz=640)

CLASS_NAMES_LIST = ['onion', 'damaged', 'rotten', 'sprouted']
GATE_THRESHOLD = 0.6

print('=' * 60)
print(f'{"Class":>12} {"mAP50":>8} {"mAP50-95":>10} {"Precision":>10} {"Recall":>8}')
print('-' * 60)

ap50 = metrics.box.ap50
ap = metrics.box.ap
p = metrics.box.p
r = metrics.box.r

for i, name in enumerate(CLASS_NAMES_LIST):
    print(f'{name:>12} {ap50[i]:>8.4f} {ap[i].mean():>10.4f} {p[i]:>10.4f} {r[i]:>8.4f}')

print('-' * 60)
print(f'{"ALL":>12} {metrics.box.map50:>8.4f} {metrics.box.map:>10.4f} {metrics.box.mp:>10.4f} {metrics.box.mr:>8.4f}')
print('=' * 60)

onion_map50 = float(ap50[0])
gate_pass = onion_map50 >= GATE_THRESHOLD
status = '✅ PASS' if gate_pass else '❌ FAIL'
print(f'\n🚦 GO/NO-GO: onion mAP50 = {onion_map50:.4f} → {status}')
if not gate_pass:
    print('   Consider: more data, more epochs, or annotation cleanup.')

## Step 7 — Confusion Matrix

In [ ]:
from pathlib import Path
from IPython.display import Image, display

# YOLO auto-saves confusion matrix during val()
cm_path = Path('/content/runs/detect/onion_yolo_v1/confusion_matrix.png')
cm_norm = Path('/content/runs/detect/onion_yolo_v1/confusion_matrix_normalized.png')

# Try the val run output folder too
val_dirs = sorted(Path('/content/runs/detect').glob('val*'), reverse=True)
for vd in val_dirs:
    candidate = vd / 'confusion_matrix_normalized.png'
    if candidate.exists():
        cm_norm = candidate
        cm_path = vd / 'confusion_matrix.png'
        break

if cm_norm.exists():
    print('Normalized Confusion Matrix:')
    display(Image(filename=str(cm_norm), width=600))
elif cm_path.exists():
    print('Confusion Matrix:')
    display(Image(filename=str(cm_path), width=600))
else:
    print('⚠️ Confusion matrix not found. Check runs/detect/ for val output.')

## Step 8 — Demo Asset Inference (pick best 3 photos for slides)

In [ ]:
from pathlib import Path
from IPython.display import Image, display
import random

best_model = YOLO('/content/runs/detect/onion_yolo_v1/weights/best.pt')

# Pick sample images from val set
val_images = sorted(Path('/content/dataset/images/val').glob('*.jpg'))
samples = random.sample(val_images, min(6, len(val_images)))

results = best_model.predict(
    source=[str(s) for s in samples],
    imgsz=640,
    conf=0.25,
    save=True,
    name='demo_assets',
)

# Display annotated results
demo_dir = Path('/content/runs/detect/demo_assets')
for img_path in sorted(demo_dir.glob('*.jpg'))[:3]:
    print(f'\n--- {img_path.name} ---')
    display(Image(filename=str(img_path), width=500))

## Step 9 — Verify model.names & Download

**Critical**: verify the model's internal class mapping matches our contract:
`{0: 'onion', 1: 'damaged', 2: 'rotten', 3: 'sprouted'}`

In [ ]:
# Verify model.names matches contract
expected = {0: 'onion', 1: 'damaged', 2: 'rotten', 3: 'sprouted'}
actual = best_model.names

print('Expected:', expected)
print('Actual:  ', actual)

if actual == expected:
    print('\n✅ model.names matches contract — safe to ship!')
else:
    print('\n❌ MISMATCH! Fix data.yaml class order before retraining.')
    for k in expected:
        if actual.get(k) != expected[k]:
            print(f'  id {k}: expected "{expected[k]}" but got "{actual.get(k)}"')

In [ ]:
# Download best.pt as onion_yolo.pt
import shutil
from google.colab import files

src = '/content/runs/detect/onion_yolo_v1/weights/best.pt'
dst = '/content/onion_yolo.pt'
shutil.copy(src, dst)

print(f'Model size: {Path(dst).stat().st_size / 1e6:.1f} MB')
print('Downloading onion_yolo.pt ...')
files.download(dst)

## Step 10 — Save eval_report.json for slides

In [ ]:
import json
from datetime import datetime, timezone

eval_report = {
    'model': 'onion_yolo.pt',
    'base': 'yolo11n.pt',
    'evaluated_at': datetime.now(timezone.utc).isoformat(),
    'epochs': 100,
    'imgsz': 640,
    'seed': 42,
    'map50': float(metrics.box.map50),
    'map50_95': float(metrics.box.map),
    'precision': float(metrics.box.mp),
    'recall': float(metrics.box.mr),
    'per_class': {},
    'go_no_go': {
        'gate': 'onion mAP50 >= 0.6',
        'onion_map50': float(ap50[0]),
        'pass': bool(ap50[0] >= 0.6),
    },
}

for i, name in enumerate(CLASS_NAMES_LIST):
    eval_report['per_class'][name] = {
        'map50': round(float(ap50[i]), 4),
        'precision': round(float(p[i]), 4),
        'recall': round(float(r[i]), 4),
    }

report_path = '/content/eval_report.json'
with open(report_path, 'w') as f:
    json.dump(eval_report, f, indent=2)

print(json.dumps(eval_report, indent=2))
files.download(report_path)

---

## ✅ Next Steps (Local Machine)

After downloading `onion_yolo.pt` and `eval_report.json`:

```bash
# 1. Place model artifact
cp ~/Downloads/onion_yolo.pt ml/models/onion_yolo.pt
cp ~/Downloads/eval_report.json ml/models/eval_report.json

# 2. Install ultralytics in backend venv
.venv/bin/pip install ultralytics

# 3. Flip to real inference
# In .env: DEMO_MODE=false

# 4. Restart backend
bash scripts/start-backend.sh

# 5. Test end-to-end
curl -X POST -F 'image=@docs/demo/sample_images/sample_onions_01.jpg' \
     http://127.0.0.1:8000/api/analyze
# Verify: is_demo=false, sane counts, latency ~100-300ms

# 6. Rollback if needed
# In .env: DEMO_MODE=true
```